# Problem 3. 두 손글씨 숫자의 합 맞히기

## 문제 개요
문서 자동화 팀은 손으로 쓴 숫자 두 개가 나란히 적힌 칸에서 **두 숫자의 합**을 읽어내려 합니다.
그런데 학습용 이미지에는 각 숫자가 무엇인지 기록되어 있지 않고 **합만** 적혀 있습니다.
이미지를 보고 두 숫자의 합(0~18)을 예측하세요.

## 파일 구조
```
Problem3/
├── problem3.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train/               숫자 쌍 이미지 10,000장 (28×56, pair_xxxxx.png)
    ├── test/                숫자 쌍 이미지 4,000장
    ├── digits/              숫자 하나짜리 이미지 1,000장 (28×28)
    ├── train.csv            filename, sum
    ├── digits.csv           filename, digit
    └── sample_submission.csv
```

## 데이터
| 파일 | 설명 |
|---|---|
| `train/`, `train.csv` | 왼쪽·오른쪽에 숫자가 하나씩 있는 28×56 흑백 이미지와 **두 숫자의 합** |
| `test/` | 같은 형식의 이미지, 합 없음 |
| `digits/`, `digits.csv` | 숫자 하나짜리 28×28 이미지 1,000장과 **그 숫자** (0~9) |

### 데이터 특징
- 숫자 쌍 이미지는 항상 왼쪽 절반(열 0~27)과 오른쪽 절반(열 28~55)에 숫자가 하나씩 있습니다.
- 합 분포는 9를 중심으로 산 모양입니다 (0과 18은 드묾).
- 출처: MNIST 손글씨 숫자 (Yann LeCun 외, CC BY-SA 3.0)를 이어 붙여 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem3/`)에 **`submission.csv`** 로 저장

## 평가
**Accuracy (정확도)** — 합이 정확히 맞은 이미지의 비율.

## 초기 코드 (베이스라인)
아래 베이스라인은 28×56 이미지 전체로 합(19개 클래스)을 바로 분류합니다. 동작은 하지만 점수가 낮습니다.

In [1]:
import os, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
torch.set_num_threads(os.cpu_count() or 4)
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv"); digits = pd.read_csv(f"{DATA_DIR}/digits.csv"); test_files = sorted(os.listdir(f"{DATA_DIR}/test"))
load = lambda folder, names: np.stack([np.asarray(Image.open(f"{DATA_DIR}/{folder}/{n}"), dtype=np.float32) / 255. for n in names])
XP, XD, XT = load("train", train.filename), load("digits", digits.filename), load("test", test_files)      # 쌍(28x56), 숫자(28x28), test 쌍
YS, YD = train["sum"].values, digits.digit.values
assert XP.shape[1:] == (28, 56) and XT.shape[1:] == (28, 56) and XD.shape[1:] == (28, 28) and len(XT) == 4000
MU, SD = XP.mean(), XP.std(); norm = lambda a: torch.tensor((a - MU) / SD, dtype=torch.float32)
pair_halves = lambda a: (norm(a).unsqueeze(1)[:, :, :, :28], norm(a).unsqueeze(1)[:, :, :, 28:])             # 왼쪽/오른쪽 숫자 (N,1,28,28)
print("train 쌍", XP.shape, "| digits", XD.shape, "| test 쌍", XT.shape)

train 쌍 (10000, 28, 56) | digits (1000, 28, 28) | test 쌍 (4000, 28, 56)


In [2]:
IDX = torch.tensor(np.add.outer(np.arange(10), np.arange(10)).ravel())                    # (a,b) -> a+b

def make_cnn(w=24):                                                                         # 숫자 하나(28x28) 분류 CNN: Conv-BN-ReLU x2 + MaxPool 블록 3개
    blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(),
                                     nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(blk(1, w), blk(w, 2 * w), blk(2 * w, 4 * w), nn.AdaptiveAvgPool2d(2), nn.Flatten(), nn.Dropout(0.3), nn.Linear(4 * w * 4, 10))

def shift(x, s=2):                                                                          # 최대 2픽셀 이동 증강
    dx, dy = np.random.randint(-s, s + 1, 2); return torch.roll(x, (int(dy), int(dx)), (2, 3))

def sum_dist(pL, pR):                                                                       # 두 숫자의 확률 -> 합(0~18)의 확률 (합성곱)
    q = (pL.unsqueeze(2) * pR.unsqueeze(1)).flatten(1)
    return torch.zeros(len(pL), 19).index_add_(1, IDX, q)

@torch.no_grad()
def digit_proba(m, X):
    m.eval(); return torch.cat([F.softmax(m(X[i:i + 1000]), 1) for i in range(0, len(X), 1000)])

def predict_sum_proba(m, a):                                                                # 쌍 이미지 -> 합 확률 (N,19)
    L, R = pair_halves(a); return sum_dist(digit_proba(m, L), digit_proba(m, R))

def fit_weak(L, R, ys, Xd, yd, epochs, seed=0, bs=128, lr=2e-3, lam=1.0):
    """합 라벨 약지도: 숫자 분류기 하나를 왼쪽/오른쪽에 공유하고, 합의 확률에 대해 합 라벨로 학습 + 숫자 라벨(1,000장) 손실을 함께 사용"""
    torch.manual_seed(seed); np.random.seed(seed); m = make_cnn()
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=5e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ((len(L) + bs - 1) // bs)); ys = torch.tensor(ys); yd = torch.tensor(yd)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(L)); t0 = time.time()
        for i in range(0, len(L), bs):
            b = perm[i:i + bs]; n = len(b); lg = m(torch.cat([shift(L[b]), shift(R[b])]))
            q = sum_dist(F.softmax(lg[:n], 1), F.softmax(lg[n:], 1)); loss_sum = -torch.log(q[torch.arange(n), ys[b]] + 1e-8).mean()
            db = torch.randint(0, len(Xd), (64,)); loss_dig = F.cross_entropy(m(shift(Xd[db])), yd[db], label_smoothing=0.05)
            loss = loss_sum + lam * loss_dig; opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if ep < 2 or ep == epochs - 1 or (ep + 1) % 5 == 0: print(f"  epoch {ep + 1}/{epochs} {time.time() - t0:.1f}s sum-loss {loss_sum.item():.3f} digit-loss {loss_dig.item():.3f}", flush=True)
    return m

In [ ]:
EPOCHS, SEEDS = 20, 1                    # 에폭당 약 10~17초(CPU, 10,000쌍). 홀드아웃 비교(2,000쌍): 숫자 1,000장만 0.9720 -> 합 라벨 약지도 0.9860
RUN_VALIDATION = False                   # True 면 쌍 2,000장을 떼어 검증 정확도를 다시 측정(약 +5분)
Xd = norm(XD).unsqueeze(1)
if RUN_VALIDATION:
    perm = np.random.RandomState(42).permutation(len(XP)); va, trn = perm[:2000], perm[2000:]
    Ltr, Rtr = pair_halves(XP[trn]); mv = fit_weak(Ltr, Rtr, YS[trn], Xd, YD, EPOCHS)
    print("검증 합 정확도:", float((predict_sum_proba(mv, XP[va]).argmax(1).numpy() == YS[va]).mean()))
# ---- 전체 train 쌍 + 숫자 1,000장으로 학습, test 합 확률을 시드 평균하여 argmax ----
L_all, R_all = pair_halves(XP); t0 = time.time(); probs = []
for sd in range(SEEDS):
    m = fit_weak(L_all, R_all, YS, Xd, YD, EPOCHS, seed=sd); probs.append(predict_sum_proba(m, XT))
Q = torch.stack(probs).mean(0); pred = Q.argmax(1).numpy()
print(f"학습+예측 {time.time() - t0:.0f}s | 예측 합 분포:", np.bincount(pred, minlength=19).tolist())
print("train 합 분포(비율 비교용):", (np.bincount(YS, minlength=19) / len(YS) * len(pred)).round(0).astype(int).tolist())

In [ ]:
submission = pd.DataFrame({"filename": test_files, "sum": pred.astype(int)})
submission.to_csv("submission.csv", index=False)

## 제출 형식
`submission.csv`

| filename | sum |
|---|---|
| test_00000.png | 7 |
| test_00001.png | 12 |

- 컬럼명 `filename`, `sum`, sum은 0~18 정수, `data/test/`의 모든 이미지 포함
- 포맷이 틀리면 0점 처리됩니다.

In [ ]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()
assert list(sub.columns) == ["filename", "sum"] and len(sub) == 4000 and sub["sum"].between(0, 18).all() and sub["sum"].dtype.kind == "i"
assert sorted(sub.filename) == sorted(os.listdir(f"{DATA_DIR}/test"))